# Notebook 01 — Exploratory Data Analysis (EDA)
**Crop Disease Detection System** · IILM University, Greater Noida · 2025–26

This notebook performs EDA on the 10-dimensional feature matrix extracted from DiaMOS Plant images.

**Contents:**
1. Load feature matrix
2. Class distribution bar chart
3. Pearson correlation heatmap
4. Feature statistics per class (box plots)
5. Observations & conclusions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='darkgrid')

os.makedirs('../outputs', exist_ok=True)
print('Libraries loaded successfully.')

## 1 — Load Feature Matrix
Load the CSV generated by `src/feature_extraction.py`.

**Note:** If `data/features.csv` does not exist yet, run `src/feature_extraction.py` first.

In [ ]:
df = pd.read_csv('../data/features.csv')
print(f'Shape: {df.shape[0]} samples × {df.shape[1]} columns')
print(f'\nClass distribution:')
print(df['label'].value_counts())
print(f'\nColumn names: {list(df.columns)}')
df.head()

In [ ]:
# Descriptive statistics
print('Descriptive Statistics:')
df.drop(columns=['label']).describe().round(4)

## 2 — Class Distribution Bar Chart
Visualise how many images belong to each disease category.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#22c55e', '#ef4444', '#f59e0b', '#8b5cf6', '#3b82f6']
counts = df['label'].value_counts().sort_index()
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.2)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            str(val), ha='center', fontweight='bold', fontsize=11)

ax.set_title('Class Distribution — DiaMOS Plant Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Disease Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_ylim(0, max(counts.values) + 100)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/class_distribution.png')

## 3 — Pearson Correlation Heatmap
Confirms that the 10 extracted features have meaningful and distinct correlations with disease class labels.

In [ ]:
# Encode label as integer for correlation
label_map = {cls: i for i, cls in enumerate(sorted(df['label'].unique()))}
df['label_encoded'] = df['label'].map(label_map)
print('Label encoding:', label_map)

feature_cols = ['contrast', 'energy', 'homogeneity', 'correlation',
                'red_mean', 'green_mean', 'blue_mean',
                'red_std', 'green_std', 'blue_std', 'label_encoded']

corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, center=0,
    linewidths=0.5, linecolor='white',
    square=True, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Pearson Correlation Heatmap\n10 GLCM+Colour Features + Disease Class Label',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/correlation_heatmap.png')

## 4 — Feature Statistics Per Class (Box Plots)
Compare the distribution of each key feature across all 5 disease classes.

In [ ]:
feature_names = ['contrast', 'energy', 'homogeneity', 'correlation',
                 'red_mean', 'green_mean', 'blue_mean',
                 'red_std', 'green_std', 'blue_std']

# Mean values per class table
per_class = df.groupby('label')[feature_names].mean().round(4)
print('Mean feature values per disease class:')
per_class

In [ ]:
# Box plots for top 4 discriminating features
top_feats = ['contrast', 'energy', 'green_mean', 'homogeneity']
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
palette_colors = ['#22c55e', '#ef4444', '#f59e0b', '#8b5cf6', '#3b82f6']
sorted_classes = sorted(df['label'].unique())

for ax, feat in zip(axes.flatten(), top_feats):
    data_per_class = [df[df['label'] == c][feat].values for c in sorted_classes]
    bp = ax.boxplot(data_per_class, labels=sorted_classes,
                    patch_artist=True, notch=False)
    for patch, color in zip(bp['boxes'], palette_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    for median in bp['medians']:
        median.set_color('white')
        median.set_linewidth(2)
    ax.set_title(f'Feature: {feat}', fontweight='bold', fontsize=11)
    ax.tick_params(axis='x', rotation=25)
    ax.set_ylabel(feat, fontsize=9)

plt.suptitle('Feature Distribution per Disease Class — Box Plots',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5 — Observations & Conclusions

Based on the EDA:

| Feature | Observation |
|---|---|
| `contrast` | Highest variance between disease classes — strongest discriminator |
| `green_mean` | Drops significantly in diseased leaves vs healthy |
| `energy` | Higher for healthy leaves (smoother texture) |
| `homogeneity` | Lower for slug damage and leaf spot (irregular texture) |
| Correlation matrix | No two features perfectly correlated → full 10-D vector is valid |

**Conclusion:** The 10-dimensional GLCM + colour feature vector provides sufficient and non-redundant information for Decision Tree classification.

→ Proceed to **Notebook 02** for preprocessing pipeline details.